# Data preprocessing


In [3]:
import pandas as pd
import numpy as np

- load data

In [4]:
URL = 'diabetes.csv'
df = pd.read_csv(URL)

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    float64
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    float64
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(4), int64(5)
memory usage: 54.1 KB
None


## Looking for missing values

In [5]:
print(df.isnull().any())

Pregnancies                 False
Glucose                     False
BloodPressure               False
SkinThickness               False
Insulin                     False
BMI                         False
DiabetesPedigreeFunction    False
Age                         False
Outcome                     False
dtype: bool


- -> Are we SURE?

In [6]:
print(df.describe())

       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  121.677083      72.389323      29.089844  141.753906   
std       3.369578   30.464161      12.106039       8.890820   89.100847   
min       0.000000   44.000000      24.000000       7.000000   14.000000   
25%       1.000000   99.750000      64.000000      25.000000  102.500000   
50%       3.000000  117.000000      72.000000      28.000000  102.500000   
75%       6.000000  140.250000      80.000000      32.000000  169.500000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    32.434635                  0.471876   33.240885    0.348958  
std      6.880498                  0.331329   11.760232    0.476951  
min     18.200000                  

## We dont have missing values , 
## BUT we have mistaken one's (some values dosn't make sense)

- for example BloodPressure = 0 (??)

In [7]:
print("Number of rows with 0 values for each varieble")
for col in df.columns:
    missing_rows = df.loc[df[col]==0].shape[0]
    print(col + ": " + str(missing_rows))

Number of rows with 0 values for each varieble
Pregnancies: 111
Glucose: 0
BloodPressure: 0
SkinThickness: 0
Insulin: 0
BMI: 0
DiabetesPedigreeFunction: 0
Age: 0
Outcome: 500


# Fixing Values

## Fix Implausible Blood Pressure Values

- A diastolic blood pressure below 40 mmHg is not physiologically compatible with a living, conscious patient, so these entries are almost certainly measurement or data-entry errors. We flag them as missing (NaN) and replace them with the column median.

In [8]:
mask_bp = df['BloodPressure'] < 40
df.loc[mask_bp, 'BloodPressure'] = np.nan
df['BloodPressure'] = df['BloodPressure'].fillna(df['BloodPressure'].median())

## Fix Outlier in Skin Thickness

- A triceps skinfold thickness of 99 mm is far outside the normal clinical range (the next highest value is 63 mm), strongly suggesting a data entry error. We flag it as missing and impute it with the column median.

In [9]:
mask_st = df['SkinThickness'] > 80
df.loc[mask_st, 'SkinThickness'] = np.nan
df['SkinThickness'] = df['SkinThickness'].fillna(df['SkinThickness'].median())

## Cap Extreme Insulin Values (Winsorizing)

- A few insulin readings (up to 846) are extremely high but not medically impossible in insulin-resistant diabetic patients, so we avoid deleting or fully replacing them. Instead, we cap any value above the 99th percentile at that threshold to reduce the influence of extreme outliers while keeping the data.

In [10]:
cap_insulin = df['Insulin'].quantile(0.99)
df['Insulin'] = np.where(df['Insulin'] > cap_insulin, cap_insulin, df['Insulin'])

## Verify the Cleaned Columns

In [11]:
df[['BloodPressure', 'SkinThickness', 'Insulin']].describe()

,BloodPressure,SkinThickness,Insulin
count,768.000000,768.000000,768.000000
mean,72.605469,28.997396,140.558854
std,11.714524,8.524525,82.340999
min,40.000000,7.000000,14.000000
25%,64.000000,25.000000,102.500000
50%,72.000000,28.000000,102.500000
75%,80.000000,32.000000,169.500000
max,122.000000,63.000000,519.900000


# Data standardization

In [12]:
from sklearn import preprocessing

In [15]:
df_scaled = preprocessing.scale(df)

df_scaled = pd.DataFrame(df_scaled, columns=df.columns)

In [16]:
df_scaled['Outcome'] = df['Outcome']
df = df_scaled

- ckeck if it worked

In [20]:
print(df.describe().loc[['mean','max','std'],].round(2).abs())

      Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
mean         0.00     0.00           0.00           0.00     0.00  0.00   
max          3.91     2.54           4.22           3.99     4.61  5.04   
std          1.00     1.00           1.00           1.00     1.00  1.00   

      DiabetesPedigreeFunction   Age  Outcome  
mean                      0.00  0.00     0.35  
max                       5.88  4.06     1.00  
std                       1.00  1.00     0.48  


# Set training,testing and validation data